# 05 — ML scoring
Scores the newest feature row for each meter using an approved model.

In [ ]:
import os
import re
from pyspark.ml import PipelineModel
from pyspark.sql import Window, functions as F

catalog = os.getenv('AIDP_CATALOG', 'aidp_poc')
if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', catalog):
    raise ValueError('AIDP_CATALOG must be a simple Spark identifier')
model_version = os.getenv('MODEL_VERSION', 'approved-v1')
model_uri = os.getenv('MODEL_URI', '/Workspace/models/smart_meter_next_kwh/approved-v1')
silver_table = f'{catalog}.silver.interval_reading'
predictions_table = f'{catalog}.ml.meter_predictions'
model = PipelineModel.load(model_uri)

base = spark.table(silver_table).where("quality_code = 'ACTUAL'")
window = Window.partitionBy('meter_id').orderBy('interval_start_utc')
latest = (base.select('meter_id', 'interval_start_utc', 'temperature_c', 'voltage_v', 'tariff_code', 'consumption_kwh')
    .withColumn('lag_1_kwh', F.lag('consumption_kwh', 1).over(window))
    .withColumn('lag_4_kwh', F.lag('consumption_kwh', 4).over(window))
    .withColumn('lag_96_kwh', F.lag('consumption_kwh', 96).over(window))
    .withColumn('rolling_4_kwh', F.avg('consumption_kwh').over(window.rowsBetween(-4, -1)))
    .withColumn('hour', F.hour('interval_start_utc'))
    .withColumn('day_of_week', F.dayofweek('interval_start_utc'))
    .withColumn('is_weekend', F.when(F.dayofweek('interval_start_utc').isin(1, 7), 1).otherwise(0))
    .where('lag_96_kwh IS NOT NULL')
    .withColumn('rn', F.row_number().over(Window.partitionBy('meter_id').orderBy(F.col('interval_start_utc').desc())))
    .where('rn = 1').drop('rn', 'consumption_kwh'))

scored = model.transform(latest).select(F.lit(model_version).alias('model_version'), 'meter_id', 'interval_start_utc', F.greatest(F.lit(0.0), F.col('prediction')).alias('prediction_kwh'), F.current_timestamp().alias('scored_at'))
scored.createOrReplaceTempView('scored_batch')
spark.sql(f'MERGE INTO {predictions_table} t USING scored_batch s ON t.model_version=s.model_version AND t.meter_id=s.meter_id AND t.interval_start_utc=s.interval_start_utc WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *')
print(f'Scored {scored.count()} meter readings with model {model_version}.')